# Modeling

Packages and setup

In [ ]:
# Imports & settings
from foodcast.imports import *
notebook_settings() 
os.chdir(PROJECT_ROOT)
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, DATA_DIR_3_4, DATA_DIR_3_5, DATA_DIR_3_6, DATA_DIR_3_7, DATA_DIR_3_8, _ = DATA_DIR_3_x
from foodcast.tools.rolling import rolling_window_avg, add_interday_variables, add_intraday_variables, unroll, season_from_month
from foodcast.tools.takeout import takeout
from foodcast.tools.labeling_functions import rename_items_by_modifications, rename_items
takeout = takeout + '|take and|take n\''

# Load Data
location_ids = location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
data = load_all_res_3_4_ai()

def to_list(s):
    try:
        val = ast.literal_eval(s)
        return val if isinstance(val, list) else [val]
    except (ValueError, SyntaxError):
        print("Error")
        print(s)
        return np.nan
    
animal_product_categories = [
    'lamb','beef_or_pork_burger','sausage','meatballs','bacon',
    'ground_meat','breakfast_sausage_patty',
    'chunked_beef_or_pork','pulled_pork',
    'unfried_chicken','fried_chicken',
    'savory_dairy','sweet_dairy','egg']

item_names_categories = {}
grouping_mappings = {}
for loc_id in location_ids:
    
    if loc_id in ['EMBVNVD207CC6','C0BE4NDSW26QN','75WYSXR9QBK5M','V3Q26BHF3SE2H','LBZEEFSBJNB3Z',
                  'SAFK7ND1HR6XS','CB2KHY1C2G9PT','S8MT0YGD2KTN9','LFZFT3VASXPED','1SQPTEGYPH0GA',
                  '9XKJD8DQTH559','LQ5EH4BKGV61T','78AY09MVJVTYE']:
    
        # targeted = pd.read_csv(Path('scripts') / 'labeling' / 'ai_targeted' / f'{loc_id}_1.csv', converters={'animal_categories': to_list})
        # for cat in animal_product_categories:
        #     item_names_categories[(loc_id,cat)] = (
        #         targeted
        #         .explode('animal_categories')
        #         .query('animal_categories == @cat')
        #         .item_name
        #         .drop_duplicates()
        #         .reset_index(drop=True))
        
        groups = pd.read_csv(Path('scripts') / 'labeling' / 'ai_grouping' / f'{loc_id}_1.csv', converters={'variants': to_list})
        grouping_mappings[loc_id] = (
            groups
            .assign(length = lambda df: df['variants'].str.len())
            .query('length > 0')
            .explode('variants')
            .rename(columns={'canonical_name':'item_name'})
            .set_index('variants')
            .item_name)

# with open(Path('data') / '3_data_parquet_relabeled' / 'animal_categories_items.pkl', 'wb') as f:
#     pickle.dump(item_names_categories, f)
with open(Path('data') / '3_data_parquet_relabeled' / 'grouping_mappings.pkl', 'wb') as f:
    pickle.dump(grouping_mappings, f)